In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import fastf1
import os
import requests
import tqdm


In [2]:
def get_fp1_best(FP1):

    laps = FP1.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [3]:
def get_fp2_best(FP2):

    laps = FP2.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [4]:
def get_fp3_best(FP3):

    laps = FP3.laps.pick_quicklaps()

    best = laps.groupby("Driver")["LapTime"].min().dt.total_seconds().reset_index()

    return best

In [5]:
def get_fp2_longrun(FP2):

    laps = FP2.laps.pick_quicklaps()

    stints = laps.groupby(["Driver", "Stint"])

    longrun = (
        stints.filter(lambda x: len(x) >= 5)
        .groupby("Driver")["LapTime"]
        .mean()
        .dt.total_seconds()
        .reset_index()
    )

    return longrun

In [6]:
def get_sector_times (FP2):

    laps = FP2.laps

    sector_times = laps.groupby("Driver")[[
    "Sector1Time",
    "Sector2Time",
    "Sector3Time"
    ]].mean().apply(lambda x: x.dt.total_seconds()).reset_index()

    return sector_times

In [7]:
def get_average_laptime (FP2):

    laps = FP2.laps

    avg_lap = laps.groupby("Driver")["LapTime"].mean().dt.total_seconds().reset_index()

    return avg_lap

In [8]:
def get_stint(FP2):

    laps = FP2.laps

    stints = (
        laps.groupby(["Driver","Stint"])
        .size()
        .reset_index(name="LapCount")
    )

    longest_stint = (
        stints.groupby("Driver")["LapCount"]
        .max()
        .reset_index(name="LongestStint")
    )

    return longest_stint

In [9]:
def get_tyre_compound (FP2):

    laps = FP2.laps

    tyres = laps.groupby("Driver")["Compound"].agg(lambda x: x.mode()[0]).reset_index()

    return tyres

In [10]:
def get_starting_grid_position (Result):

    grid = Result.results[["Abbreviation","GridPosition"]]
    grid.rename(columns={"Abbreviation":"Driver"},inplace=True)
    return grid

In [11]:
def get_season_points(Result, race):

    data = []

    for r in range(1, race):

        race_points = Result.results[["Abbreviation","Points"]]

        data.append(race_points)

    df = pd.concat(data)

    points = df.groupby("Abbreviation")["Points"].sum().reset_index()

    points.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return points

In [12]:
def get_season_avg_finish(Result, race):

    results_list = []

    for r in range(1, race):

        results = Result.results[["Abbreviation", "Position"]]

        results_list.append(results)

    df = pd.concat(results_list)

    season_avg_finish = df.groupby("Abbreviation")["Position"].mean().reset_index()

    season_avg_finish.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return season_avg_finish

In [13]:
def get_constructor_position(season, race):

    all_results = []

    for r in range(1, race):

        session = fastf1.get_session(season, r, "R")
        session.load()

        results = session.results[["Abbreviation", "TeamName", "Points"]]

        all_results.append(results)

    df = pd.concat(all_results)

    team_points = (
        df.groupby("TeamName")["Points"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )

    team_points["ConstructorPosition"] = team_points.index + 1

    driver_constructor = df[["Abbreviation", "TeamName"]].drop_duplicates()

    driver_constructor = driver_constructor.merge(
        team_points[["TeamName", "ConstructorPosition"]],
        on="TeamName",
        how="left"
    )

    constructor_position = driver_constructor[["Abbreviation", "TeamName", "ConstructorPosition"]]

    constructor_position.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return constructor_position

In [14]:
def get_driver_dnf_rate(Result, race):

    results_list = []

    for r in range(1, race):

        results = Result.results[["Abbreviation","Status"]]

        results_list.append(results)

    df = pd.concat(results_list)

    df["DNF"] = df["Status"] != "Finished"

    dnf_rate = (
        df.groupby("Abbreviation")["DNF"]
        .mean()
        .reset_index()
        .rename(columns={"DNF": "DriverDNFRate"})
    )

    dnf_rate.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return dnf_rate

In [15]:
def get_team_dnf_rate(season, race):

    results_list = []

    for r in range(1, race):

        session = fastf1.get_session(season, r, "R")
        session.load()

        results = session.results[["Abbreviation", "TeamName", "Status"]]

        results_list.append(results)

    df = pd.concat(results_list)

    df["DNF"] = df["Status"] != "Finished"

    team_dnf_rate = (
        df.groupby("TeamName")["DNF"]
        .mean()
        .reset_index()
        .rename(columns={"DNF": "TeamDNFRate"})
    )

    driver_team = df[["Abbreviation", "TeamName"]].drop_duplicates()

    driver_team = driver_team.merge(
        team_dnf_rate,
        on="TeamName",
        how="left"
    )

    dnf_rate = driver_team[["Abbreviation", "TeamName", "TeamDNFRate"]]

    dnf_rate.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return dnf_rate

In [16]:
def get_team_avg_finish(season, race):

    results_list = []

    for r in range(1, race):

        session = fastf1.get_session(season, r, "R")
        session.load()

        results = session.results[["Abbreviation", "TeamName", "Position"]]

        results_list.append(results)

    df = pd.concat(results_list)

    team_avg_finish = (
        df.groupby("TeamName")["Position"]
        .mean()
        .reset_index()
        .rename(columns={"Position": "TeamAverageFinish"})
    )

    driver_team = df[["Abbreviation", "TeamName"]].drop_duplicates()

    driver_team = driver_team.merge(
        team_avg_finish,
        on="TeamName",
        how="left"
    )

    team_avg_finish = driver_team[["Abbreviation", "TeamName", "TeamAverageFinish"]]

    team_avg_finish.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return team_avg_finish


In [17]:
def get_constructor_championship_position(season, race):

    results_list = []

    for r in range(1, race):

        session = fastf1.get_session(season, r, "R")
        session.load()

        results = session.results[["Abbreviation", "TeamName", "Points"]]

        results_list.append(results)

    df = pd.concat(results_list)

    team_points = (
        df.groupby("TeamName")["Points"]
        .sum()
        .reset_index()
        .sort_values("Points", ascending=False)
    )

    team_points["ConstructorChampionshipPosition"] = range(1, len(team_points) + 1)

    driver_team = df[["Abbreviation", "TeamName"]].drop_duplicates()

    driver_team = driver_team.merge(
        team_points[["TeamName", "ConstructorChampionshipPosition"]],
        on="TeamName",
        how="left"
    )

    constructor_championship_position = driver_team[
        ["Abbreviation", "TeamName", "ConstructorChampionshipPosition"]
    ]

    constructor_championship_position.rename(
        columns={"Abbreviation": "Driver"}, inplace=True
    )

    return constructor_championship_position

In [18]:
def get_rolling_team_performance(season, race, window=3):

    results_list = []

    start_round = max(1, race - window)

    for r in range(start_round, race):

        session = fastf1.get_session(season, r, "R")
        session.load()

        results = session.results[["Abbreviation", "TeamName", "Position"]]

        results_list.append(results)

    df = pd.concat(results_list)

    df["Position"] = pd.to_numeric(df["Position"], errors="coerce")

    rolling_performance = (
        df.groupby("TeamName")["Position"]
        .mean()
        .reset_index()
        .rename(columns={"Position": "RollingTeamPerformance"})
    )

    driver_team = df[["Abbreviation", "TeamName"]].drop_duplicates()

    driver_team = driver_team.merge(
        rolling_performance,
        on="TeamName",
        how="left"
    )

    team = driver_team[["Abbreviation", "TeamName", "RollingTeamPerformance"]]

    team.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    return team

In [19]:
def get_qualifying_data(Qualify):

    qual = Qualify.results[["Abbreviation", "Q3"]].copy()

    qual.rename(columns={"Abbreviation": "Driver"}, inplace=True)

    qual["Q3"] = qual["Q3"].dt.total_seconds()

    return qual

In [20]:
def get_race_results(Result):

    race = Result.results[["Abbreviation","Position"]]
    race.rename(columns={"Abbreviation":"Driver"},inplace=True)

    return race

In [21]:
season = 2022

race = 5

FP1 = fastf1.get_session(season, race, 'FP1')
FP1.load()
FP2 = fastf1.get_session(season, race, 'FP2')
FP2.load()
FP3 = fastf1.get_session(season, race, 'FP3')
FP3.load()

Result = fastf1.get_session(season, race, 'R')
Result.load()

Qualify = fastf1.get_session(season, race, 'Q')
Qualify.load()


print(f"Done Loading Season {season} & Race {race}")

fp1_best = get_fp1_best(FP1)
fp2_best = get_fp2_best(FP2)
fp3_best = get_fp3_best(FP3)

longrun = get_fp2_longrun(FP2)
sector_times = get_sector_times(FP2)
average_laptime = get_average_laptime(FP2)
stint = get_stint(FP2)
tyre_compound = get_tyre_compound(FP2)
starting_grid_position = get_starting_grid_position(FP2)
season_points = get_season_points(Result,race)
season_avg_finish = get_season_avg_finish(Result,race)

constructor_position = get_constructor_position(season, race)
driver_dnf_rate = get_driver_dnf_rate(Result, race)
team_dnf_rate = get_team_dnf_rate(season, race)
team_avg_finish = get_team_avg_finish(season, race)
constructor_championship_position = get_constructor_championship_position(season, race)
rolling_team_performance = get_rolling_team_performance(season, race, window=3)

qualifying_data = get_qualifying_data(Qualify)
race_results = get_race_results(Result)

print("Calculation Complete")

fp1_best.set_index("Driver",inplace=True)
fp2_best.set_index("Driver",inplace=True)
fp3_best.set_index("Driver",inplace=True)
longrun.set_index("Driver",inplace=True)
sector_times.set_index("Driver",inplace=True)
average_laptime.set_index("Driver", inplace=True)
stint.set_index("Driver",inplace=True)
tyre_compound.set_index("Driver",inplace=True)
starting_grid_position.set_index("Driver",inplace=True)
season_points.set_index("Driver",inplace=True)
season_avg_finish.set_index("Driver",inplace=True)
constructor_position.set_index("Driver",inplace=True)
driver_dnf_rate.set_index("Driver",inplace=True)
team_dnf_rate.set_index("Driver",inplace=True)
team_avg_finish.set_index("Driver",inplace=True)
constructor_championship_position.set_index("Driver",inplace=True)
rolling_team_performance.set_index("Driver",inplace=True)
qualifying_data.set_index("Driver",inplace=True)
race_results.set_index("Driver",inplace=True)

print("Setting Index Complete")

df = fp1_best.copy()
df.rename(columns={"LapTime":"FP1_BestTime(s)"},inplace=True)
df[["FP2_BestTime(s)"]] = fp2_best[["LapTime"]]
df[["FP3_BestTime(s)"]] = fp3_best[["LapTime"]]
df[["LongRun(s)"]] = longrun[["LapTime"]]
df[["Sector1Time(s)","Sector2Time(s)","Sector3Time(s)"]] = sector_times[["Sector1Time","Sector2Time","Sector3Time"]]
df[["Average_LapTime(s)"]] = average_laptime[["LapTime"]]
df[["Longest_Stint"]] = stint[["LongestStint"]]
df[["Tyre"]] = tyre_compound[["Compound"]]
df[["Starting_Grid_Position"]] = starting_grid_position[["GridPosition"]]
df[["Season_Points"]] = season_points[["Points"]]
df[["Season_Average_Finish"]] = season_avg_finish[["Position"]]
df[["Constructor_Position","Team"]] = constructor_championship_position[["ConstructorChampionshipPosition","TeamName"]]
df[["Driver_DNF_Rate"]] = driver_dnf_rate[["DriverDNFRate"]]
df[["Team_DNF_Rate"]] = team_dnf_rate[["TeamDNFRate"]]
df[["Team_Average_Finish"]] = team_avg_finish[["TeamAverageFinish"]]
df[["Rolling_Team_Performance"]] = rolling_team_performance[["RollingTeamPerformance"]]
df[["Qualifying_Time(s)"]] = qualifying_data[["Q3"]]
df[["Final_Standing"]] = race_results[["Position"]]

print("Colunmns Complete")

fp1_best.reset_index(inplace=True)
fp2_best.reset_index(inplace=True)
fp3_best.reset_index(inplace=True)
longrun.reset_index(inplace=True)
sector_times.reset_index(inplace=True)
average_laptime.reset_index(inplace=True)
stint.reset_index(inplace=True)
tyre_compound.reset_index(inplace=True)
starting_grid_position.reset_index(inplace=True)
season_points.reset_index(inplace=True)
season_avg_finish.reset_index(inplace=True)
constructor_position.reset_index(inplace=True)
driver_dnf_rate.reset_index(inplace=True)
team_dnf_rate.reset_index(inplace=True)
team_avg_finish.reset_index(inplace=True)
constructor_championship_position.reset_index(inplace=True)
rolling_team_performance.reset_index(inplace=True)
qualifying_data.reset_index(inplace=True)
race_results.reset_index(inplace=True)
df.reset_index(inplace=True)

print("Resetting Index Complete")

req         WARNING 	DEFAULT CACHE ENABLED! (3.87 GB) /home/satyam/.cache/fastf1


core           INFO 	Loading data for Miami Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '3', '4', '5', '6', '10', '11', '14', '16', '18', '20', '22', '23', '24', '31', '44', '47', '55', '63', '77']
core           INFO 	Loading data for Miami Grand Prix - Practice 2 [v3.8.1]
req            INFO 	Using cached d

Done Loading Season 2022 & Race 5


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_

Calculation Complete
Setting Index Complete
Colunmns Complete
Resetting Index Complete


/tmp/ipykernel_424607/2097842272.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  race.rename(columns={"Abbreviation":"Driver"},inplace=True)


In [22]:
df

,Driver,FP1_BestTime(s),FP2_BestTime(s),FP3_BestTime(s),LongRun(s),Sector1Time(s),Sector2Time(s),Sector3Time(s),Average_LapTime(s),Longest_Stint,...,Season_Points,Season_Average_Finish,Constructor_Position,Team,Driver_DNF_Rate,Team_DNF_Rate,Team_Average_Finish,Rolling_Team_Performance,Qualifying_Time(s),Final_Standing
0,ALB,91.854,91.710,91.501,NaN,37.254824,39.738350,32.025450,107.805294,7.0,...,8.0,9.0,10,Williams,0.0,0.625,14.250000,14.166667,NaN,9.0
1,ALO,92.884,90.372,91.036,94.7014,37.591471,40.227800,31.524800,108.717882,8.0,...,0.0,11.0,6,Alpine,0.0,0.500,11.875000,13.166667,NaN,11.0
2,BOT,93.773,NaN,91.885,NaN,NaN,NaN,NaN,NaN,NaN,...,24.0,7.0,5,Alfa Romeo,0.0,0.250,10.125000,10.833333,89.475,7.0
3,GAS,91.498,90.547,91.901,94.7390,36.821765,40.035474,29.834789,105.837063,8.0,...,0.0,18.0,7,AlphaTauri,1.0,0.500,12.250000,11.666667,89.690,18.0
4,HAM,91.956,90.179,91.890,NaN,39.833667,42.785353,32.088765,104.443083,8.0,...,32.0,6.0,3,Mercedes,0.0,0.125,5.750000,6.500000,89.625,6.0
5,LAT,95.637,92.913,92.376,NaN,40.121909,41.051769,33.209692,110.641300,7.0,...,0.0,14.0,10,Williams,0.0,0.625,14.250000,14.166667,NaN,14.0
6,LEC,91.098,90.044,90.498,NaN,38.326778,40.775850,31.689550,104.917533,8.0,...,72.0,2.0,1,Ferrari,0.0,0.250,6.875000,8.666667,88.796,2.0
7,MAG,92.559,90.921,91.227,NaN,36.790813,39.029947,31.701263,105.643812,9.0,...,0.0,16.0,8,Haas F1 Team,1.0,0.500,11.142857,12.400000,NaN,16.0
8,MSC,94.945,91.587,91.050,NaN,36.173647,40.178050,32.250200,108.004125,9.0,...,0.0,15.0,8,Haas F1 Team,0.0,0.500,11.142857,12.400000,NaN,15.0
9,NOR,92.615,90.535,91.594,NaN,36.785118,41.205100,37.084700,108.858312,7.0,...,0.0,19.0,4,McLaren,1.0,0.250,10.625000,9.333333,89.750,19.0


In [24]:
fastf1.Cache.enable_cache("f1_cache")

In [25]:
dataset = []

for season in range(2019, 2026):

    schedule = fastf1.get_event_schedule(season)

    for race in schedule["RoundNumber"]:

        try:

            print(f"Loading Season {season} Round {race}")

            FP1 = fastf1.get_session(season, race, "FP1"); FP1.load()
            FP2 = fastf1.get_session(season, race, "FP2"); FP2.load()
            FP3 = fastf1.get_session(season, race, "FP3"); FP3.load()

            Result = fastf1.get_session(season, race, "R"); Result.load()
            Qualify = fastf1.get_session(season, race, "Q"); Qualify.load()

            # Feature extraction
            fp1_best = get_fp1_best(FP1)
            fp2_best = get_fp2_best(FP2)
            fp3_best = get_fp3_best(FP3)

            longrun = get_fp2_longrun(FP2)
            sector_times = get_sector_times(FP2)
            average_laptime = get_average_laptime(FP2)
            stint = get_stint(FP2)
            tyre_compound = get_tyre_compound(FP2)
            starting_grid_position = get_starting_grid_position(FP2)

            season_points = get_season_points(Result, race)
            season_avg_finish = get_season_avg_finish(Result, race)

            constructor_position = get_constructor_position(season, race)
            driver_dnf_rate = get_driver_dnf_rate(Result, race)
            team_dnf_rate = get_team_dnf_rate(season, race)
            team_avg_finish = get_team_avg_finish(season, race)
            constructor_championship_position = get_constructor_championship_position(season, race)
            rolling_team_performance = get_rolling_team_performance(season, race)

            qualifying_data = get_qualifying_data(Qualify)
            race_results = get_race_results(Result)

            # Set index
            dfs = [
                fp1_best, fp2_best, fp3_best, longrun, sector_times,
                average_laptime, stint, tyre_compound, starting_grid_position,
                season_points, season_avg_finish, constructor_position,
                driver_dnf_rate, team_dnf_rate, team_avg_finish,
                constructor_championship_position, rolling_team_performance,
                qualifying_data, race_results
            ]

            for d in dfs:
                d.set_index("Driver", inplace=True)

            # Build dataset
            df = fp1_best.copy()
            df.rename(columns={"LapTime": "FP1_BestTime(s)"}, inplace=True)

            df["FP2_BestTime(s)"] = fp2_best["LapTime"]
            df["FP3_BestTime(s)"] = fp3_best["LapTime"]
            df["LongRun(s)"] = longrun["LapTime"]

            df[["Sector1Time(s)", "Sector2Time(s)", "Sector3Time(s)"]] = sector_times[
                ["Sector1Time", "Sector2Time", "Sector3Time"]
            ]

            df["Average_LapTime(s)"] = average_laptime["LapTime"]
            df["Longest_Stint"] = stint["LongestStint"]
            df["Tyre"] = tyre_compound["Compound"]
            df["Starting_Grid_Position"] = starting_grid_position["GridPosition"]

            df["Season_Points"] = season_points["Points"]
            df["Season_Average_Finish"] = season_avg_finish["Position"]

            df["Constructor_Position"] = constructor_championship_position["ConstructorChampionshipPosition"]
            df["Team"] = constructor_championship_position["TeamName"]

            df["Driver_DNF_Rate"] = driver_dnf_rate["DriverDNFRate"]
            df["Team_DNF_Rate"] = team_dnf_rate["TeamDNFRate"]
            df["Team_Average_Finish"] = team_avg_finish["TeamAverageFinish"]

            df["Rolling_Team_Performance"] = rolling_team_performance["RollingTeamPerformance"]

            df["Qualifying_Time(s)"] = qualifying_data["Q3"]
            df["Final_Standing"] = race_results["Position"]

            df["Season"] = season
            df["Round"] = race

            df.reset_index(inplace=True)

            dataset.append(df)

            print(f"Finished Season {season} Round {race}")

        except Exception as e:
            print(f"Skipping Season {season} Round {race}: {e}")

core           INFO 	Loading data for Australian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Loading Season 2019 Round 1


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to

Skipping Season 2019 Round 1: No objects to concatenate
Loading Season 2019 Round 2


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to

...

core           INFO 	Loading data for Brazilian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Skipping Season 2019 Round 19: any API: 500 calls/h
Loading Season 2019 Round 20


core           INFO 	Loading data for Abu Dhabi Grand Prix - Practice 1 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Skipping Season 2019 Round 20: any API: 500 calls/h
Loading Season 2019 Round 21
Skipping Season 2019 Round 21: any API: 500 calls/h


RateLimitExceededError: any API: 500 calls/h